# Chapter 11: Memory in Agents + MCP
## Topic 3: What MCP Is and Why It Matters

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every tool built across Chapter 10 and Chapter 11 so far — `validate_fd_reference`, `search_knowledge_base`, `lookup_product_catalog`, `check_sender_history` — is wired directly into this project's own single agent, using this project's own bespoke `if/elif` dispatch logic (Chapter 10 Topic 1-2). This works well for one project with one agent — but it doesn't generalize at all: if a second, different agent (or a completely different application) wanted to reuse `check_sender_history`, it would need its own separate, hand-written integration, duplicating the same dispatch and schema logic all over again.
- **MCP (Model Context Protocol)** exists specifically to solve this N×M problem — a term this project's own earlier notes (Chapter 7's design-decision discussion) already used precisely: without a standard, N different agents each needing M different tools would require N×M separate, bespoke integrations. MCP defines a standardized protocol so that any MCP-compatible tool server can be connected to any MCP-compatible agent, without either side needing custom, bespoke integration code — reducing N×M integration work to roughly N+M (each tool server implements the protocol once; each agent implements the protocol once).
- The core distinction this topic establishes precisely: this project's current tools are exposed *directly* to one specific agent's dispatch logic. An MCP-wrapped tool is instead exposed through a standardized *server* that speaks a common protocol — any properly MCP-compatible agent (this project's own agent, a different agent, a completely different team's agent) could connect to that same server and use the same tool, without needing to know anything about this project's own internal dispatch conventions.


### 2. Internal Working — Step by Step

**What MCP actually standardizes, conceptually, building directly on Chapter 10's tool-calling mechanics:**

1. **MCP standardizes how a tool describes itself** — conceptually the same information already present in this project's own tool schemas (Chapter 10 Topic 4: name, description, input schema) but expressed in a standard, protocol-defined format any MCP-compatible client can parse and understand, rather than a project-specific dictionary structure only this project's own dispatch code knows how to read.
2. **MCP standardizes how a client (an agent) discovers what tools are available from a given server** — rather than this project's `TOOLS` list being a hardcoded Python list only this codebase knows about, an MCP server can be *queried* by any compatible client to discover its available tools dynamically, at connection time, rather than requiring the tool list to be known and hardcoded in advance by every agent that wants to use it.
3. **MCP standardizes the actual request-and-response mechanics for invoking a tool** — conceptually the same request/dispatch/result cycle already built in Chapter 10 Topic 1-2 (a structured request naming the tool and its arguments, a real execution step, a structured result returned) but happening over a standard protocol between two potentially separate processes or systems, rather than within one Python program's own in-memory function calls.
4. **MCP standardizes access to "resources"** (not just callable tools) — a capability distinct from tool-calling that this project hasn't needed yet, but is directly relevant to Chapter 7 Topic 3's earlier observation that MCP's Resources capability could be more immediately useful than its Tools capability for exposing static, referenceable content (like policy or product reference files) in a standard way, separate from invoking an active capability like a database lookup.


### 3. How This Is Implemented in This Project

- This project's `validate_fd_reference`, `check_sender_history`, and `lookup_product_catalog` are all excellent, well-designed *candidates* for MCP-wrapping (Chapter 10 Topic 3's and Chapter 11 Topic 2's design principles — structured output, clear found/not-found distinctions, curated fields — are exactly what makes a tool a good MCP citizen too, since MCP doesn't change what makes a tool well-designed, only how it's exposed to potentially multiple different agents).
- As already noted precisely in this project's own earlier design discussion (Chapter 7 Topic 3), MCP adoption is not currently justified at this project's actual N=1 agent scale — there's only one agent (this project's own `run_agent()`), so the N×M problem MCP solves doesn't yet apply in a way that would produce real, measured benefit. The identified triggers for reconsidering — a second consumer of these tools, a need for process isolation, or genuine ops-facing tool reuse across projects — are the concrete, evidence-based conditions worth watching for, rather than adopting MCP preemptively based on its general reputation.
- This topic (and the two that follow, building and connecting a minimal MCP server) exist specifically as a hands-on learning exercise to genuinely understand the mechanism — building a small, real MCP server for this project's own tools is valuable for understanding the protocol concretely, independent of whether this specific project's current single-agent scale actually justifies adopting it in production yet.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **MCP adoption is a genuine architectural decision with real trade-offs, not a strictly-better upgrade over direct tool integration.** Exposing a tool through an MCP server, potentially as a genuinely separate process, adds real communication overhead (network or inter-process communication) compared to a direct, in-process function call — this project's own current direct-dispatch approach (Chapter 10 Topic 1-2) is measurably faster for a single-agent, single-process use case, and MCP's benefit (reusability, standardization) only pays off once there's genuinely more than one consumer to justify that overhead.
- **Standardization has a real cost in flexibility.** A bespoke, project-specific tool schema (this project's current approach) can be shaped exactly to this project's own needs with no compromise; conforming to a standard protocol's conventions may occasionally require adapting a tool's natural shape to fit the protocol's expectations — a real, if usually modest, cost worth weighing against the standardization benefit.
- **Security and trust boundaries change meaningfully once a tool is exposed via a standard, potentially network-accessible protocol** rather than only being callable from within one trusted, single-process Python program — an MCP server needs its own authentication, authorization, and input-validation discipline (directly connecting to Chapter 10 Topic 1's dispatch-layer validation principle, now needing to be enforced at a genuine process or network boundary, not just within trusted, single-process code).
- **Debugging a tool-calling issue becomes genuinely more complex once client and server are separate processes or systems** — Chapter 10 Topic 2's message-list debugging technique (inspect exactly what was sent) still applies conceptually, but now spans a client-server boundary, potentially requiring visibility into both sides rather than a single Python program's own internal state.
- **Monitoring and deployment:** an MCP server, if genuinely adopted, becomes its own deployable, monitorable service with its own uptime, latency, and error-rate concerns — a new operational surface this project's current, simpler direct-integration approach doesn't have to manage at all.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Direct tool integration (this project's current, correct choice) vs MCP-wrapped tools:** direct integration is simpler, faster, and has no additional infrastructure to deploy or maintain — the right choice while this project genuinely has only one agent consuming its own tools. MCP becomes worth the added complexity specifically once a second, genuinely separate consumer needs the same tools, not before.
- **Which tools would be the first, best candidates for MCP-wrapping, if and when that trigger is actually met:** tools already well-designed by this chapter's own principles (structured output, clear input/output contracts, minimal project-specific coupling) — `check_sender_history` and `validate_fd_reference` are both strong candidates precisely because their design is already clean and self-contained, unlike a tool that happened to be written with implicit, project-specific assumptions baked in that would need to be untangled before it could be cleanly exposed to a different consumer.
- **Adopting MCP for its Resources capability specifically, ahead of its Tools capability:** given this project's own earlier observation (Chapter 7 Topic 3) that Resources may be more immediately useful than Tools at current scale — worth revisiting as a narrower, lower-commitment first step into MCP adoption, rather than treating "adopt MCP" as an all-or-nothing decision covering both capabilities at once.


### 6. Alternatives and When to Use Each

- **Direct, bespoke tool integration (this project's current approach):** the right choice at N=1 agent scale — simple, fast, no added infrastructure, and appropriate until a genuine second consumer exists.
- **MCP adoption:** the right choice once at least one of the concrete triggers identified earlier in this project (a second consumer, a need for process isolation, genuine ops-facing tool reuse across projects) is actually met — not before, based purely on MCP's general reputation as a standard worth adopting.
- **A different, custom-built internal API layer (rather than adopting the MCP standard specifically) for sharing tools across multiple internal consumers:** a viable alternative to MCP specifically, worth considering if the actual consumers are all internal, trusted, and tightly coupled already — MCP's standardization benefit matters most when consumers might be genuinely external, loosely coupled, or built by different teams entirely.


### 7. Common Mistakes and Production Failures

- Adopting MCP preemptively, before any genuine second consumer or other concrete trigger exists, incurring real added complexity and communication overhead for a benefit that doesn't yet apply at the project's actual current scale.
- Treating MCP as a strictly-better replacement for direct tool integration rather than understanding it as a genuine trade-off — faster, simpler direct integration for a single consumer versus standardized, reusable, but more overhead-laden integration for multiple consumers.
- Exposing a tool via MCP without applying the same input-validation and access-control discipline (Chapter 10 Topic 1) now needed at a genuine process or network boundary, rather than only within trusted, single-process code.
- Choosing a poorly-designed, project-specific-assumption-laden tool as a first MCP-wrapping candidate, rather than starting with tools that are already clean, well-designed, and minimally coupled to this project's own internal conventions.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What specific problem does MCP solve?
  A: The N×M integration problem — without a standard protocol, N different agents each needing M different tools would require N×M separate, bespoke integrations. MCP standardizes how tools describe themselves, how clients discover available tools, and how tool invocation requests and results are exchanged, so any MCP-compatible tool server can connect to any MCP-compatible agent without custom integration code on either side.

- Q: Why isn't MCP adoption currently justified for this project's own agent?
  A: This project currently has only one agent consuming its own tools — the N×M problem MCP solves doesn't yet apply in a way that produces real, measured benefit, since there's no second consumer to share integration work with. Direct, bespoke tool integration (this project's current approach) is simpler and faster for this single-consumer case.

**Intermediate**

- Q: What are the concrete triggers that would justify reconsidering MCP adoption for this project?
  A: A second, genuinely separate consumer needing the same tools (making the N×M integration savings real rather than theoretical), a need for process isolation (running a tool in a separate, sandboxed process from the agent itself), or genuine ops-facing tool reuse across different projects or teams. Any of these would mean MCP's standardization benefit actually outweighs its added communication overhead and complexity.

- Q: How does MCP's standardized tool description relate to this project's own existing tool schemas (Chapter 10 Topic 4)?
  A: Conceptually, they carry the same information — a name, a description, and an input schema — but MCP expresses this in a protocol-standard format any compatible client can parse, rather than a project-specific Python dictionary structure only this codebase's own dispatch logic understands. The underlying design principles for writing a *good* description (Chapter 10 Topic 4's when-to-call and when-not-to-call guidance) apply identically either way — MCP standardizes the format, not the content-quality principles.

**Advanced**

- Q: A teammate argues this project should adopt MCP now, in anticipation of future growth, rather than waiting for a concrete trigger. How do you respond?
  A: This is exactly the kind of premature-adoption risk this chapter's own migration-trigger discipline (mirroring Chapter 6's vector-database migration principle) warns against — MCP adds real communication overhead and a new operational surface (a deployable, monitorable server) that isn't justified without an actual second consumer or other concrete trigger. The right approach is building tools well (as this chapter's design principles already ensure) so they'd be straightforward to wrap in MCP later if a genuine trigger appears, rather than paying MCP's overhead now for a benefit that may or may not ever materialize.

- Q: How would you decide which of this project's tools should be the first candidates for MCP-wrapping, once a genuine trigger is met?
  A: Prioritize tools that are already well-designed by this chapter's own standards — structured output, clear input/output contracts, minimal implicit coupling to this project's own internal conventions or assumptions. `check_sender_history` and `validate_fd_reference` are strong candidates precisely because their clean design (Chapter 9 Topic 3, Chapter 11 Topic 2) means little to no rework would be needed to expose them safely and usefully to a different, external consumer.

**Scenario-based**

- Q: A second team at the same company wants to reuse this project's `check_sender_history` tool for their own, unrelated agent. Walk through how you'd evaluate whether this is the trigger that justifies MCP adoption.
  A: This is close to exactly the "second consumer" trigger this chapter identifies — worth confirming the second team's actual technical needs (do they need the same data shape, the same access-control model, similar latency requirements) align well enough that a shared, standardized interface genuinely reduces total integration work compared to each team maintaining its own separate, bespoke connection to this data. If so, wrapping this specific tool in MCP, rather than the project's entire tool set at once, is a reasonable, narrowly-scoped first step — validating the approach on one genuinely-justified case before considering broader adoption.

**Follow-up questions to expect:**

- "What would the security and access-control model look like for an MCP server exposing customer data across two different teams' agents?"
- "How would you measure whether MCP adoption actually reduced total integration effort, versus just moving the complexity somewhere else?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The N×M-to-N+M reduction MCP provides is a specific instance of a very general software engineering pattern: standardized interfaces/protocols reducing pairwise integration cost.** This same shape of problem and solution appears constantly in software engineering — anywhere multiple independent producers and consumers need to interoperate, a shared standard (an API convention, a data format, a protocol) reduces the total integration burden from growing with every new pairing to growing much more slowly.
- **The "don't adopt a standard until a concrete trigger justifies it" discipline is the same evidence-based migration principle already applied twice elsewhere in this notebook series** (Chapter 6's vector database migration, Chapter 11 Topic 2's CSV-to-database migration) — recognizing this as a recurring, generalizable engineering discipline, not a one-off judgment call specific to any single technology decision, is a mark of genuinely transferable understanding.
- **This topic sets up Topics 4 and 5 directly:** understanding MCP's conceptual value here is what makes actually building a minimal MCP server (Topic 4) and connecting an agent to it (Topic 5) meaningful hands-on exercises, rather than mechanical steps followed without understanding why the mechanism exists or when it would genuinely matter in production.

### 10. Quick Revision Summary (for last-minute interview prep)

> MCP solves the N×M integration problem — without a standard, N agents each needing M tools require N×M separate, bespoke integrations; MCP standardizes tool description, discovery, and invocation so any compatible client can connect to any compatible server, reducing this to roughly N+M. This project's current tools are all directly, bespokely integrated into one single agent (Chapter 10's dispatch pattern), which is the right choice at this project's actual N=1 agent scale — MCP's standardization benefit doesn't yet apply in a way that outweighs its real added communication overhead and new operational surface. The concrete triggers worth watching for before reconsidering — a genuine second consumer, a need for process isolation, or real ops-facing tool reuse across teams — mirror exactly the same evidence-based migration discipline already established twice elsewhere in this notebook series (Chapter 6's vector database migration, Chapter 11 Topic 2's CSV-to-database migration): stay simple until a measured, real need justifies the added complexity, not before. Well-designed tools, following this chapter's own established principles, are exactly what would make wrapping them in MCP straightforward later, if and when a genuine trigger actually appears.


### Module 1: The N×M Integration Cost, Computed for Real Scenarios

Model direct, bespoke integration cost as a genuine combinatorial calculation across growing numbers of agents and tools.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: N x M integration cost, computed concretely
# ------------------------------------------------------------------

def direct_integration_cost(num_agents: int, num_tools: int) -> int:
    """With DIRECT, bespoke integration (this project's current
    approach), each agent needs its OWN integration code for EVERY
    tool it wants to use -- a genuine N x M relationship."""
    return num_agents * num_tools


def standardized_protocol_cost(num_agents: int, num_tools: int) -> int:
    """With a STANDARDIZED PROTOCOL (MCP), each tool implements the
    protocol ONCE (regardless of how many agents will use it), and
    each agent implements a protocol CLIENT once (regardless of how
    many tools it will use) -- a genuine N + M relationship."""
    return num_agents + num_tools


print("=" * 70)
print("MODULE 1: DIRECT (N x M) vs STANDARDIZED (N + M) INTEGRATION COST")
print("=" * 70)

scenarios = [
    ("This project's ACTUAL current scale", 1, 4),   # 1 agent, 4 tools (Ch10-11's tools)
    ("A second team adopts the same tools", 2, 4),
    ("Three teams, tool set grows to 6", 3, 6),
    ("Five teams, tool set grows to 10", 5, 10),
]

print(f"{'Scenario':<38} | {'Agents':>6} | {'Tools':>5} | {'Direct (NxM)':>12} | {'Standard (N+M)':>14}")
print("-" * 90)
for label, agents, tools in scenarios:
    direct = direct_integration_cost(agents, tools)
    standard = standardized_protocol_cost(agents, tools)
    print(f"{label:<38} | {agents:>6} | {tools:>5} | {direct:>12} | {standard:>14}")

print("\nAt THIS PROJECT'S ACTUAL current scale (1 agent, 4 tools):")
current_direct = direct_integration_cost(1, 4)
current_standard = standardized_protocol_cost(1, 4)
print(f"  Direct integration cost: {current_direct}")
print(f"  Standardized protocol cost: {current_standard}")
standardization_verdict = "CHEAPER" if current_standard < current_direct else "MORE EXPENSIVE"
print(f"  Standardization would be {standardization_verdict}"
      f" by {abs(current_direct - current_standard)} integration point(s)")

print("\nThis is the CONCRETE, computed version of this project's own")
print("earlier stated conclusion (Chapter 7 Topic 3): at N=1 agent, MCP's")
print("standardization overhead is NOT justified by the integration-cost")
print("math -- the savings only become real once N genuinely grows past 1.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: DIRECT (N x M) vs STANDARDIZED (N + M) INTEGRATION COST
Scenario                               | Agents | Tools | Direct (NxM) | Standard (N+M)
------------------------------------------------------------------------------------------
This project's ACTUAL current scale    |      1 |     4 |            4 |              5
A second team adopts the same tools    |      2 |     4 |            8 |              6
Three teams, tool set grows to 6       |      3 |     6 |           18 |              9
Five teams, tool set grows to 10       |      5 |    10 |           50 |             15

At THIS PROJECT'S ACTUAL current scale (1 agent, 4 tools):
  Direct integration cost: 4
  Standardized protocol cost: 5
  Standardization would be MORE EXPENSIVE by 1 integration point(s)

This is the CONCRETE, computed version of this project's own
earlier stated conclusion (Chapter 7 Topic 3): at N=1 agent, MCP's
standardization overhead is NOT justified by the integration-cost
math -- the savings

### Module 2: The Crossover Point — Where Standardization Actually Starts Paying Off

Find the exact point, computed rather than assumed, where standardized-protocol cost becomes cheaper than direct integration cost, holding tool count fixed at this project's real current tool count.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The crossover point -- computed, not assumed
# ------------------------------------------------------------------

FIXED_TOOL_COUNT = 4  # this project's actual current tool count
                       # (validate_fd_reference, search_knowledge_base,
                       #  lookup_product_catalog, check_sender_history)

print("=" * 70)
print(f"CROSSOVER POINT (tool count fixed at {FIXED_TOOL_COUNT}, varying agent count)")
print("=" * 70)
print(f"{'Num Agents':>10} | {'Direct (NxM)':>12} | {'Standard (N+M)':>14} | {'Standardization wins?':>22}")
print("-" * 65)

crossover_agent_count = None
for num_agents in range(1, 8):
    direct = direct_integration_cost(num_agents, FIXED_TOOL_COUNT)
    standard = standardized_protocol_cost(num_agents, FIXED_TOOL_COUNT)
    wins = standard < direct
    if wins and crossover_agent_count is None:
        crossover_agent_count = num_agents
    print(f"{num_agents:>10} | {direct:>12} | {standard:>14} | {str(wins):>22}")

print(f"\nCrossover point: standardization becomes cheaper starting at "
      f"{crossover_agent_count} agent(s), given {FIXED_TOOL_COUNT} tools.")
print(f"This project currently has 1 agent -- BELOW this crossover point,")
print(f"confirming direct integration is genuinely the cheaper, correct")
print(f"choice RIGHT NOW, not just a simplification for this notebook.")

print("\nModule 2 complete. Run Module 3 next.")


CROSSOVER POINT (tool count fixed at 4, varying agent count)
Num Agents | Direct (NxM) | Standard (N+M) |  Standardization wins?
-----------------------------------------------------------------
         1 |            4 |              5 |                  False
         2 |            8 |              6 |                   True
         3 |           12 |              7 |                   True
         4 |           16 |              8 |                   True
         5 |           20 |              9 |                   True
         6 |           24 |             10 |                   True
         7 |           28 |             11 |                   True

Crossover point: standardization becomes cheaper starting at 2 agent(s), given 4 tools.
This project currently has 1 agent -- BELOW this crossover point,
confirming direct integration is genuinely the cheaper, correct
choice RIGHT NOW, not just a simplification for this notebook.

Module 2 complete. Run Module 3 next.


### Module 3: Bespoke Dispatch vs Generic Discovery — A Concrete Code Comparison

Contrast this project's actual hardcoded if/elif dispatch (Chapter 10) against a generic, schema-driven dispatch mechanism that could work with ANY tool following a standard description format, without per-tool hardcoded logic.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Bespoke dispatch vs generic, schema-driven discovery
# ------------------------------------------------------------------

# --- THIS PROJECT'S ACTUAL APPROACH: hardcoded if/elif dispatch ---
def bespoke_dispatch(tool_name: str, tool_input: dict) -> dict:
    """Exactly this project's real Chapter 10 pattern -- adding a NEW
    tool requires editing THIS function's code directly."""
    if tool_name == "validate_fd_reference":
        return {"result": f"looked up FD {tool_input.get('reference_number')}"}
    elif tool_name == "check_sender_history":
        return {"result": f"checked history for {tool_input.get('sender_email')}"}
    # adding a THIRD tool means adding ANOTHER elif branch here --
    # the dispatch function itself must be edited for every new tool
    else:
        return {"error": "unknown_tool"}


# --- STANDARDIZED APPROACH: tools register themselves, generic dispatch ---
TOOL_REGISTRY = {}

def register_tool(name: str, handler_function):
    """Standardized registration -- a tool ANNOUNCES itself, rather
    than a central dispatch function needing to know about it in advance."""
    TOOL_REGISTRY[name] = handler_function

def generic_dispatch(tool_name: str, tool_input: dict) -> dict:
    """GENERIC dispatch -- this function's code NEVER needs to change
    when a new tool is added, as long as the new tool registers itself
    following the same convention. This is the CONCEPTUAL mechanism
    an MCP client uses to invoke ANY discovered tool generically."""
    handler = TOOL_REGISTRY.get(tool_name)
    if handler is None:
        return {"error": "unknown_tool"}
    return handler(tool_input)


def validate_fd_reference_handler(tool_input: dict) -> dict:
    return {"result": f"looked up FD {tool_input.get('reference_number')}"}

def check_sender_history_handler(tool_input: dict) -> dict:
    return {"result": f"checked history for {tool_input.get('sender_email')}"}

def lookup_product_catalog_handler(tool_input: dict) -> dict:
    return {"result": f"looked up product {tool_input.get('product_name')}"}


register_tool("validate_fd_reference", validate_fd_reference_handler)
register_tool("check_sender_history", check_sender_history_handler)
register_tool("lookup_product_catalog", lookup_product_catalog_handler)


print("=" * 70)
print("BESPOKE DISPATCH vs GENERIC, REGISTRATION-BASED DISPATCH")
print("=" * 70)

print("\nBespoke dispatch (this project's actual current approach):")
bespoke_result_1 = bespoke_dispatch("validate_fd_reference", {"reference_number": "BJ2019FD7717"})
bespoke_result_2 = bespoke_dispatch("lookup_product_catalog", {"product_name": "Smart Saver FD"})
print(f"  {bespoke_result_1}")
print(f"  {bespoke_result_2}")
print("  ^ NOTE: lookup_product_catalog was NEVER added to bespoke_dispatch's")
print("    if/elif chain -- it falls through to 'unknown_tool', exactly")
print("    demonstrating that EVERY new tool requires editing this")
print("    function's own code directly.")

print("\nGeneric, registration-based dispatch (MCP's conceptual mechanism):")
generic_result_1 = generic_dispatch("validate_fd_reference", {"reference_number": "BJ2019FD7717"})
generic_result_2 = generic_dispatch("lookup_product_catalog", {"product_name": "Smart Saver FD"})
print(f"  {generic_result_1}")
print(f"  {generic_result_2}")
print("  ^ This SAME dispatch function correctly handled a THIRD tool")
print("    that was never mentioned inside generic_dispatch's own code --")
print("    it only needed to be REGISTERED, exactly the discovery")
print("    mechanism MCP standardizes at the protocol level.")

print("\nModule 3 complete. All Chapter 11 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 11 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  MCP solves a genuine N x M -> N + M integration cost reduction --
  demonstrated with REAL computed numbers, not just asserted.

  At THIS PROJECT'S actual scale (1 agent, 4 tools), direct integration
  is genuinely cheaper -- computed crossover point confirms standardization
  only starts paying off once agent count grows past 1.

  Bespoke dispatch (this project's real Chapter 10 pattern) requires
  EDITING the dispatch function for every new tool. Generic,
  registration-based dispatch (MCP's conceptual mechanism) does NOT --
  demonstrated concretely: the same generic function handled a tool it
  never explicitly knew about.

  MCP adoption should follow a CONCRETE, MEASURED trigger (a real
  second consumer, process isolation need, cross-team reuse) -- not
  general reputation, mirroring the same evidence-based migration
  discipline already used for the vector database (Ch6) and the
  sender-history store (Ch11 Topic 2).
""")


BESPOKE DISPATCH vs GENERIC, REGISTRATION-BASED DISPATCH

Bespoke dispatch (this project's actual current approach):
  {'result': 'looked up FD BJ2019FD7717'}
  {'error': 'unknown_tool'}
  ^ NOTE: lookup_product_catalog was NEVER added to bespoke_dispatch's
    if/elif chain -- it falls through to 'unknown_tool', exactly
    demonstrating that EVERY new tool requires editing this
    function's own code directly.

Generic, registration-based dispatch (MCP's conceptual mechanism):
  {'result': 'looked up FD BJ2019FD7717'}
  {'result': 'looked up product Smart Saver FD'}
  ^ This SAME dispatch function correctly handled a THIRD tool
    that was never mentioned inside generic_dispatch's own code --
    it only needed to be REGISTERED, exactly the discovery
    mechanism MCP standardizes at the protocol level.

Module 3 complete. All Chapter 11 Topic 3 modules done.

CHAPTER 11 TOPIC 3 -- KEY POINTS TO REMEMBER

  MCP solves a genuine N x M -> N + M integration cost reduction --
  demonst